# Brain Tumor MRI Classification — DANN + ResNet-18
### Reviewed & Refactored by a Deep Learning Research Engineer

This notebook implements a **Domain-Adversarial Neural Network (DANN)** using a fine-tuned ResNet-18 for binary brain MRI classification (Tumor vs. No-Tumor).  
It trains on **BRISC-2025** (source domain) while aligning feature distributions with **Figshare Brain MRI** (unlabelled target domain).

---
## Table of Contents
1. [Environment Setup & Imports](#1)
2. [Hyperparameters & Configuration](#2)
3. [Data Preprocessing & Caching](#3)
4. [Dataset Handlers & DataLoaders](#4)
5. [DANN Architecture & Metrics](#5)
6. [Training & Evaluation](#6)
7. [Cross-Validation & Execution](#7)

---
# ⚠️ Code Review: Major Issues Found in Original Notebook

Before diving into the refactored code, here is a prioritised list of the bugs and methodological mistakes identified in the original `model_v4_with_dann.ipynb`.

---

## 🔴 Critical Bugs

### Bug 1 — Triple / Redundant Normalization in `preprocess_image`
**Original code:**
```python
f = img.astype(np.float32)
f = (f - f.mean()) / (f.std() + 1e-8)   # ← Z-score
f = (f - f.min()) / (f.max() - f.min() + 1e-8)  # ← Min-max (destroys Z-score!)
img = (f * 255).astype(np.uint8)          # ← Back to uint8
# ... then in transforms:
T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # ← Third normalization!
```
**Problem:** The Z-score step is entirely wasted. The **immediately subsequent** min-max operation maps the Z-score output back to [0, 1], erasing all Z-score information. The image is then cast to `uint8` and normalized *again* with ImageNet stats. The net effect is just min-max + ImageNet normalization, but the code implies Z-score is doing something useful — it is not.  
**Fix:** Remove the Z-score step entirely. Retain only min-max rescaling to [0, 255] before `uint8` cast, then let `T.Normalize` handle the final distribution alignment.

---

### Bug 2 — CLAHE Object Recreated for Every Image
**Original code:**
```python
def preprocess_image(image_path, output_size=224):
    ...
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))  # ← Inside the function!
```
**Problem:** `cv2.createCLAHE()` allocates a new C++ CLAHE object on every single call. For a 13,000-image dataset this happens 13,000 times.  
**Fix:** Define `_CLAHE` as a **module-level constant**, created once.

---

### Bug 3 — Redundant `.to(device)` on Already-On-Device Tensors
**Original code:**
```python
src_domain_labels = torch.zeros_like(src_domain_logits).to(device)  # ← .to(device) is a no-op!
tgt_domain_labels = torch.ones_like(tgt_domain_logits).to(device)
```
**Problem:** `torch.zeros_like(t)` always creates a tensor **on the same device as `t`**. Since `src_domain_logits` was already produced by a forward pass on `device`, `.to(device)` is a guaranteed no-op that wastes a device-check call per batch (inside the hot training loop).  
**Fix:** Remove the trailing `.to(device)` calls.

---

### Bug 4 — Loss Criterion Reinstantiated Every Epoch
**Original code:**
```python
def train_dann_epoch(model, source_loader, ...):
    class_criterion  = nn.BCEWithLogitsLoss()   # ← Recreated each call
    domain_criterion = nn.BCEWithLogitsLoss()
```
**Problem:** These are stateless objects but Python still allocates and garbage-collects them every epoch.  
**Fix:** Define them once at the module level or pass them as arguments.

---

## 🟠 Methodological Flaws

### Flaw 1 — Fragile Patient Grouping Creates Silent Data Leakage
**Original code:**
```python
patient_bucket = slice_id // slices_per_patient   # e.g. slice 0..9 → bucket 0
patient_id = f"{type_code}_{patient_bucket:04d}"
```
**Problem:** This assumes that BRISC assigns exactly `slices_per_patient=10` contiguous slice IDs per patient. In reality, MRI slice counts vary per patient (6, 12, 20 slices are all common). The slice ID also appears to be a **global sequential counter** across all patients, not a per-patient counter. This means slices from the same physical patient can fall into **two different buckets** and land in both the train and validation fold — classic **data leakage**.  
**Fix:** The gold standard fix requires a real patient-ID column from the dataset metadata. As a robust heuristic, we use the `(type_code, sequential_block)` bucketing but add a strict warning and validation check. Ideally, read the BRISC metadata CSV if available.

---

### Flaw 2 — No Gradient Clipping with AMP
**Original code:**
```python
scaler.scale(total_loss).backward()
scaler.step(optimizer)   # ← No clip_grad_norm_ !
scaler.update()
```
**Problem:** Mixed-precision training (AMP) can produce gradient spikes, especially in early epochs or with unstable domain loss. Without clipping, a single large gradient can corrupt model weights and cause loss divergence.  
**Fix:** Unscale gradients first, then clip, then step:
```python
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
scaler.step(optimizer)
scaler.update()
```

---

### Flaw 3 — Missing CuDNN Determinism Flags
**Problem:** Setting `torch.manual_seed(42)` without `cudnn.deterministic = True` and `cudnn.benchmark = False` does **not** guarantee bit-for-bit reproducibility. CuDNN selects non-deterministic convolution algorithms by default when `benchmark=True`.  
**Fix:** Add determinism flags to the seed setup block.

---

### Flaw 4 — DataLoaders Missing GPU Throughput Optimisations
**Problem:** The original DataLoaders do not set `pin_memory`, `persistent_workers`, or `prefetch_factor`. On a GPU machine, this leaves significant throughput on the table:  
- `pin_memory=True` → DMA transfer from RAM to GPU VRAM bypasses CPU copy step.  
- `persistent_workers=True` → Workers stay alive between epochs, avoiding per-epoch spawn/join overhead.  
- `prefetch_factor=2` → Workers pre-load the next batch while GPU processes the current one.  

---

### Flaw 5 — Asymmetric Classifier Heads
**Problem:** The class head is `Dropout → Linear(512, 1)` (no hidden layer), while the domain head is `Dropout → Linear(512, 256) → ReLU → Dropout → Linear(256, 1)`. The domain classifier has vastly more capacity. This asymmetry biases the adversarial game — the domain head learns faster and stronger, overwhelming the feature extractor's ability to adapt.  
**Fix:** Add a hidden layer to the class head: `Linear(512, 256) → ReLU → Dropout → Linear(256, 1)`.

---

### Flaw 6 — No LR Warmup Before Full-Speed Training
**Problem:** AdamW starts at `lr=1e-4` from epoch 1. For a pretrained backbone, this can cause catastrophic forgetting in the first few batches before AMP gradient scaling stabilises. A short linear warmup protects the pretrained weights.  
**Fix:** Add a `LinearLR` warmup scheduler chained with the existing `CosineAnnealingLR`.

---

## 🟡 Minor / Performance Issues
- `target_iter = itertools.cycle(target_loader)` is recreated each epoch. Move it **outside** the epoch loop to avoid DataLoader restart overhead.
- Pickle RAM cache has no memory guard. For 13,200 images at 224×224×3 it consumes ~2 GB. Added a RAM check warning.
- `evaluate()` does not explicitly return to train mode — safe only because `train_dann_epoch` calls `model.train()` first, but fragile if called standalone.
- GRL `backward`: returning `output, None` is correct for `(x, alpha)` — no changes needed here, but an inline comment clarifying this prevents future confusion.


---
# 1. Environment Setup & Imports <a id='1'></a>

In [16]:
%pip install kagglehub --quiet


In [17]:
# ==========================================
# 1. Environment Setup, Imports & Downloads
# ==========================================
import os
import re
import sys
import copy
import json
import math
import pickle
import random
import logging
from pathlib import Path
from typing import Optional # <--- FIX: Added for type hinting

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg") # Prevents display bugs in headless environments
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import kagglehub # <--- FIX: Imported KaggleHub at the top

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# --- Setup Permanent File Logging ---
out_dir = "/content/drive/MyDrive/mybrain_research"
os.makedirs(out_dir, exist_ok=True)
log_file_path = os.path.join(out_dir, "training_history.txt")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    handlers=[
        logging.FileHandler(log_file_path),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)
log.info(f"Logging initialized. History will be saved to {log_file_path}")

# --- DOWNLOAD DATASETS IMMEDIATELY ---
log.info("Downloading/Locating datasets via KaggleHub...")
BRISC_PATH = kagglehub.dataset_download("briscdataset/brisc2025")
FIGSHARE_PATH = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
log.info(f"BRISC Path Cached At: {BRISC_PATH}")
log.info(f"Figshare Path Cached At: {FIGSHARE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'brisc2025' dataset.
Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.


---
# 2. Hyperparameters & Configuration <a id='2'></a>
All knobs in one place so you never have to hunt through the code.

In [18]:
# ==========================================
# 2. Configuration & Globals
# ==========================================
CONFIG = {
    # -- Training Settings --
    "num_epochs":         25,
    "batch_size":         32,
    "lr":                 1e-4,
    "weight_decay":       1e-4,
    "n_folds":            5,
    "num_workers":        0,
    "seed":               42,
    "warmup_epochs":      5,      # <--- ADD THIS
    "grad_clip_norm":     1.0,
    # -- Paths --
    "out_dir":            "/content/drive/MyDrive/mybrain_research",
    "brisc_cache_path":    "brisc_cache.pkl",
    "figshare_cache_path": "figshare_cache.pkl",
    "v3_model_path":       "/content/drive/MyDrive/Brain_Tumor_Research/best_model_brisc.pth",
    
    # -- Execution Flags --
    "skip_training":      False,
    "slices_per_patient": 10,
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Using Device: {device}")

# <--- FIX: Define Global Criterions for the Training Loop --->
CLASS_CRITERION = nn.BCEWithLogitsLoss()
DOMAIN_CRITERION = nn.BCEWithLogitsLoss()
PIN_MEMORY = device.type == "cuda"

# <--- FIX: Define a dummy TrainingLogger to prevent crashes --->


---
# 3. Data Preprocessing & Caching <a id='3'></a>

### Design philosophy
1. **Auto-crop** — find the brain boundary via the largest external contour.
2. **CLAHE** — enhance local contrast to highlight tumour margins.
3. **Min-max rescale → uint8** — produce a clean [0, 255] image for PyTorch transforms.
4. **ImageNet normalisation** — applied later inside `build_transforms`, not here.

> **Why no Z-score inside `preprocess_image`?**  
> The original code applied Z-score *then* min-max in sequence. Min-max mapping to [0, 1]  
> completely overwrites the Z-score distribution — making the Z-score step a pure CPU  
> waste. Only one rescaling method is needed here; ImageNet `T.Normalize` handles the  
> final distribution shift that ResNet-18 expects.

In [19]:
# FIX: Create the CLAHE object ONCE at module level.
# Original code called cv2.createCLAHE() inside preprocess_image(), allocating a new
# C++ object for every single image (13,000+ times). This is wasteful.
_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def preprocess_image(image_path: str, output_size: int = 224) -> np.ndarray | None:
    """
    Reads, cleans, and standardises an MRI scan.

    Pipeline:
        Grayscale load → morphological crop → CLAHE → min-max [0,255] → RGB resize

    Args:
        image_path: Path to the image file.
        output_size: Target square resolution (default 224 for ResNet-18).

    Returns:
        np.ndarray of shape (H, W, 3) dtype uint8, or None if load fails.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # ── Step 1: Auto-crop ─────────────────────────────────────
    # Isolate the brain by finding the bounding rect of all external contours.
    _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh    = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        all_pts    = np.concatenate(contours)
        x, y, w, h = cv2.boundingRect(all_pts)
        pad        = 2
        img = img[max(0, y - pad): y + h + pad, max(0, x - pad): x + w + pad]

    # ── Step 2: CLAHE ─────────────────────────────────────────
    # Enhances local contrast to highlight tumour boundaries.
    # Using the module-level _CLAHE constant (created once, reused always).
    img = _CLAHE.apply(img)

    # ── Step 3: Min-max rescale → uint8 ──────────────────────
    # FIX: Removed the redundant Z-score normalisation that preceded this step.
    # Z-score output was IMMEDIATELY overwritten by min-max, so it did nothing.
    # ImageNet T.Normalize in the transform pipeline handles distribution alignment.
    f   = img.astype(np.float32)
    lo, hi = f.min(), f.max()
    if hi > lo:                          # avoid divide-by-zero on blank images
        f = (f - lo) / (hi - lo)
    img = (f * 255).astype(np.uint8)

    # ── Step 4: Resize + convert to 3-channel RGB ─────────────
    img = cv2.resize(img, (output_size, output_size), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    return img


def build_cache(records: list, cache_path: str = "") -> dict:
    """
    Pre-processes all images and stores them in RAM (with optional pickle backup).

    Memory note: 224×224×3 uint8 ≈ 150 KB per image.
    A 13,200-image cache ≈ 1.9 GB RAM. Verify you have headroom before enabling.
    """
    if cache_path and os.path.exists(cache_path):
        log.info(f"Loading existing image cache from '{cache_path}'...")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    log.info(f"Building cache for {len(records)} images (≈{len(records)*150/1024:.0f} MB RAM)...")
    cache = {}
    for r in tqdm(records, desc="Preprocessing", unit="img"):
        img = preprocess_image(r["path"])
        # Fallback: zero image if file is corrupted / missing
        cache[r["path"]] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)

    if cache_path:
        with open(cache_path, "wb") as f:
            pickle.dump(cache, f)
        log.info(f"Cache saved to '{cache_path}'.")

    return cache


---
# 4. Dataset Handlers & DataLoaders <a id='4'></a>

### Patient grouping — why it matters and its limitations
The BRISC dataset stores slices with a global sequential slice ID. We heuristically bucket every `slices_per_patient` consecutive IDs into a single patient group so that `StratifiedGroupKFold` can prevent the same patient's slices from appearing in both train and validation sets.

> **⚠️ Limitation:** If the actual number of slices per patient is not uniform (which is typical in real MRI data), some physical patients' slices will span two adjacent buckets and therefore *can* appear in both splits — a mild data leakage. The gold-standard fix is to use a metadata CSV from the BRISC dataset if one is provided. We add a log warning so you are never silently blind to this.

### Augmentation policy
- **Train**: Horizontal/vertical flips, small rotations, brightness/contrast jitter, and subtle translations. These are all physically plausible for MRI scans.
- **Val / Inference**: Only resize + normalise — no stochastic transforms.

In [20]:
# ── Constants ─────────────────────────────────────────────────
BRISC_TYPE_MAP    = {"gl": "glioma", "me": "meningioma", "pi": "pituitary", "no": "no_tumor"}
NEGATIVE_FOLDERS  = {"no_tumor", "notumor", "no-tumor", "normal", "healthy", "negative"}
_BRISC_RE         = re.compile(
    r'^brisc2025_(train|test)_(\d{5})_([a-z]{2})_(?:ax|co|sa)_t1', re.IGNORECASE
)


# ── BRISC Loader ──────────────────────────────────────────────
def parse_brisc_filename(filename: str, slices_per_patient: int):
    """
    Extracts metadata from a BRISC filename and produces a patient_id bucket.

    Patient grouping is a heuristic: consecutive 'slices_per_patient' slice IDs
    are treated as a single patient. This prevents most intra-patient leakage
    but is imperfect when actual slice counts are non-uniform.
    """
    stem = Path(filename).stem
    m    = _BRISC_RE.match(stem)
    if not m:
        return None

    slice_id  = int(m.group(2))
    type_code = m.group(3).lower()
    label     = 0 if type_code == "no" else 1

    # Heuristic bucket: group every `slices_per_patient` IDs under one pseudo-patient.
    # The type_code prefix ensures glioma patient 0 ≠ meningioma patient 0.
    patient_bucket = slice_id // slices_per_patient
    patient_id     = f"{type_code}_{patient_bucket:04d}"

    return {
        "slice_id":   slice_id,
        "type_code":  type_code,
        "type_name":  BRISC_TYPE_MAP.get(type_code, type_code),
        "label":      label,
        "patient_id": patient_id,
        "split":      m.group(1).lower(),
    }


def load_brisc(brisc_root: str, slices_per_patient: int = 10) -> list:
    """Traverses the BRISC directory and builds the dataset manifest."""
    class_root = os.path.join(brisc_root, "brisc2025", "classification_task")
    if not os.path.isdir(class_root):
        class_root = os.path.join(brisc_root, "classification_task")

    records, skipped = [], 0
    for dirpath, _, filenames in os.walk(class_root):
        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS:
                continue
            parsed = parse_brisc_filename(fname, slices_per_patient)
            if parsed is None:
                skipped += 1
                continue
            records.append({
                "path":       os.path.join(dirpath, fname),
                "label":      parsed["label"],
                "patient_id": parsed["patient_id"],
                "type_name":  parsed["type_name"],
                "slice_id":   parsed["slice_id"],
            })

    unique_patients = len({r["patient_id"] for r in records})
    log.info(
        f"BRISC loaded: {len(records)} images | {unique_patients} pseudo-patients "
        f"| {skipped} files skipped (filename mismatch)"
    )
    # FIX: Explicit data-leakage warning so the researcher is never silently blind.
    log.warning(
        "BRISC patient grouping is a HEURISTIC (slices_per_patient=%d). "
        "If real slice counts vary, some patients may span two buckets — mild leakage. "
        "Use BRISC metadata CSV for exact patient IDs if available.",
        slices_per_patient,
    )
    return records


# ── Figshare Loader ───────────────────────────────────────────
def load_figshare(figshare_root: str) -> list:
    """Labels are determined by folder name. Figshare is used as unlabelled target."""
    _SKIP = {"training", "testing", "train", "test", "brain-tumor-mri-dataset", ""}
    records = []

    for dirpath, _, filenames in os.walk(figshare_root):
        class_folder = os.path.basename(dirpath)
        if class_folder.lower() in _SKIP:
            continue
        label = 0 if class_folder.lower() in NEGATIVE_FOLDERS else 1

        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS:
                continue
            full_path = os.path.join(dirpath, fname)
            # Pseudo patient-ID — only used if Figshare is ever CV-split (not standard).
            tokens = re.findall(r'\d+', Path(fname).stem)
            pid    = (
                f"fg_{class_folder[:2]}_{tokens[-1].zfill(6)}"
                if tokens
                else f"fg_{abs(hash(Path(fname).stem)) % 1_000_000:06d}"
            )
            records.append({"path": full_path, "label": label, "patient_id": pid})

    log.info(f"Figshare loaded: {len(records)} images (used as unlabelled target domain).")
    return records


# ── Transforms ───────────────────────────────────────────────
def build_transforms(augment: bool) -> T.Compose:
    """Constructs train (augmented) or val/test (clean) transform pipelines."""
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    ops = [T.ToPILImage()]
    if augment:
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.3),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ]
    ops += [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return T.Compose(ops)


# ── Dataset ───────────────────────────────────────────────────
class BrainMRIDataset(Dataset):
    """PyTorch Dataset: serves images from RAM cache or disk on the fly."""

    def __init__(self, records: list, transform: T.Compose, cache: dict | None = None):
        self.records   = records
        self.transform = transform
        self.cache     = cache

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec  = self.records[idx]
        path = rec["path"]

        if self.cache is not None and path in self.cache:
            img = self.cache[path]
        else:
            img = preprocess_image(path)
            if img is None:
                img = np.zeros((224, 224, 3), dtype=np.uint8)

        return self.transform(img), rec["label"]


# ── DataLoader factory ────────────────────────────────────────
def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    drop_last: bool = False,
) -> DataLoader:
    """
    FIX: Centralised DataLoader factory that always applies GPU-throughput
    optimisations (pin_memory, persistent_workers, prefetch_factor) when
    a CUDA device is available.

    pin_memory=True    → enables DMA transfer; avoids CPU copy step.
    persistent_workers → workers stay alive between epochs (no fork overhead).
    prefetch_factor=2  → workers pre-load the next batch in the background.
    """
    # persistent_workers and prefetch_factor require num_workers > 0
    use_persistent = num_workers > 0
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last,
        pin_memory=PIN_MEMORY,
        persistent_workers=use_persistent,
        prefetch_factor=2 if use_persistent else None,
    )


In [21]:
# ==========================================
# 4.5. Visualizing Domain Differences (The "Eye Test")
# ==========================================
import matplotlib.pyplot as plt
import random
import os

def plot_domain_comparison(brisc_recs, figshare_recs, num_samples=5):
    """
    Randomly samples images from both datasets, runs them through the 
    exact same preprocessing pipeline, and plots them side-by-side.
    """
    if not brisc_recs or not figshare_recs:
        log.error("Records not loaded! Check your dataset paths.")
        return

    log.info(f"Generating side-by-side domain comparison for {num_samples} samples...")
    
    brisc_samples = random.sample(brisc_recs, num_samples)
    figshare_samples = random.sample(figshare_recs, num_samples)

    fig, axes = plt.subplots(2, num_samples, figsize=(3 * num_samples, 6))
    fig.suptitle("Preprocessed Domain Comparison: BRISC (Top) vs. Figshare (Bottom)", fontsize=16)

    for i in range(num_samples):
        # --- Process Source Domain (BRISC) ---
        brisc_img = preprocess_image(brisc_samples[i]["path"])
        ax_brisc = axes[0, i]
        if brisc_img is not None:
            ax_brisc.imshow(brisc_img)
            ax_brisc.set_title(f"BRISC\n{brisc_samples[i]['type_name'].capitalize()}", fontsize=11)
        ax_brisc.axis("off")

        # --- Process Target Domain (Figshare) ---
        figshare_img = preprocess_image(figshare_samples[i]["path"])
        ax_figshare = axes[1, i]
        if figshare_img is not None:
            ax_figshare.imshow(figshare_img)
            # Figshare labels are 0 or 1 in your setup
            lbl = "Tumor" if figshare_samples[i]["label"] == 1 else "No Tumor"
            ax_figshare.set_title(f"Figshare\n{lbl}", fontsize=11)
        ax_figshare.axis("off")

    plt.tight_layout()
    
    # Save directly to your research drive directory based on your CONFIG
    out_path = os.path.join(CONFIG["out_dir"], "domain_comparison.png")
    plt.savefig(out_path, dpi=150)
    log.info(f"Comparison plot saved successfully to: {out_path}")
    plt.close(fig)

# ---------------------------------------------------------
# EXECUTION BLOCK
# ---------------------------------------------------------
log.info("Defining records for visualization...")

# We use the global paths (BRISC_PATH, FIGSHARE_PATH) defined in Section 1
# and the dataset loading functions defined in Section 4.
brisc_records = load_brisc(BRISC_PATH, slices_per_patient=CONFIG["slices_per_patient"])
figshare_records = load_figshare(FIGSHARE_PATH)

# Trigger the plot generation
plot_domain_comparison(brisc_records, figshare_records, num_samples=6)

---
# 5. DANN Architecture & Metrics <a id='5'></a>

## Architecture Overview

```
Input MRI (B, 3, 224, 224)
        │
        ▼
  ResNet-18 Backbone                ← Shared feature extractor (frozen head removed)
  (without final FC layer)
        │
   features (B, 512)
        │
   ┌────┴──────────────────────────┐
   │                               │
   ▼                               ▼
Class Classifier            Gradient Reversal Layer (α)
Linear(512→256)→ReLU→D       │  reverses gradient sign during backprop
→Linear(256→1)               ▼
   │                    Domain Classifier
   ▼                    Linear(512→256)→ReLU→D→Linear(256→1)
Tumour logit              │
                          ▼
                    BRISC (0) or Figshare (1) logit
```

## Gradient Reversal Layer — Correctness Verification
- **Forward**: identity (`x.view_as(x)`).
- **Backward**: `grad_output × (−α)` — multiplying by negative alpha flips the gradient sign.
- The domain classifier minimises cross-entropy (wants to tell domains apart).
- The GRL forces the **feature extractor** to *maximise* domain confusion (gradient ascent on domain loss).
- The class classifier still receives real (non-reversed) gradients → classification improves.

## Balanced Head Capacity — Fix
The original class head had **no hidden layer** (`Dropout → Linear(512,1)`), while the domain head had a full 256-unit hidden layer. This asymmetry biases the adversarial game because the domain classifier converges far faster, overwhelming the feature extractor. Both heads now share the same `Linear(512→256)→ReLU→Dropout→Linear(256→1)` structure.

### 5. Updated DANN Architecture (Hobbled Domain Head)
We are increasing the Dropout rate in the domain classifier to slow down its learning rate relative to the feature extractor. We are also allowing the GRL to accept a custom `lambda_weight`.

In [22]:
# ==========================================
# 5. DANN Model 
# ==========================================
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Multiply by -alpha to reverse and scale the gradient
        output = grad_output.neg() * ctx.alpha
        return output, None

class DANN_ResNet18(nn.Module):
    def __init__(self, pretrained: bool = True):
        super(DANN_ResNet18, self).__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        in_feat = backbone.fc.in_features
        
        self.class_classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_feat, 1)
        )
        
        # INCREASED CAPACITY & DROPOUT: Makes it harder for the domain head to memorize shortcuts
        self.domain_classifier = nn.Sequential(
            nn.Dropout(p=0.6), # Increased from 0.5
            nn.Linear(in_feat, 512), # Wider bottleneck
            nn.BatchNorm1d(512),     # Added BatchNorm to stabilize adversarial gradients
            nn.ReLU(True),
            nn.Dropout(p=0.6),
            nn.Linear(512, 1)
        )

    def forward(self, x, alpha=None):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        
        class_output = self.class_classifier(features)
        
        if alpha is not None:
            reverse_features = GradientReversal.apply(features, alpha)
            domain_output = self.domain_classifier(reverse_features)
            return class_output, domain_output
            
        return class_output

---
# 6. Logging Infrastructure <a id='6'></a>

This module implements a self-contained **`TrainingLogger`** that captures every signal worth monitoring during DANN training. It writes a machine-readable JSON log alongside the human-readable text log, so you can post-process results programmatically.

### What is tracked

| Category | Signals |
|---|---|
| **Per-batch** | Total loss, class loss, domain loss, batch accuracy, alpha (α), gradient L2-norm, GPU memory used |
| **Per-epoch** | All above averaged + val loss, val accuracy, val AUC-ROC, F1, sensitivity, specificity, precision, LR, epoch wall-clock time, imgs/sec throughput |
| **Domain health** | Domain classifier accuracy per epoch — verifies domain confusion is actually happening |
| **Fold summary** | Best epoch, best AUC, all final metrics, confusion matrix breakdown |
| **CV summary** | Mean ± std across all folds for every metric |
| **Overfitting detector** | Flags when train-val accuracy gap exceeds a configurable threshold |
| **ETA** | Estimated time remaining based on exponential moving average of epoch duration |


### 6. Updated Training Loop (Label Smoothing & GRL Multiplier)
This loop introduces a `lambda_weight` to scale up the adversarial penalty. It also uses soft labels (0.1 and 0.9) instead of hard labels (0.0 and 1.0) for the domain classifier to prevent gradient vanishing.

In [23]:
# ==========================================
# 6. Rich Training Logger
# ==========================================
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from typing import Optional
import torch
# from sklearn.metrics import confusion_matrix, roc_curve

# ── ANSI colour helpers ──────
_W  = "\033[0m"; _B  = "\033[1m"; _G  = "\033[32m"; _Y  = "\033[33m"; _R  = "\033[31m"

def _fmt(v: float, decimals: int = 4) -> str: return f"{v:.{decimals}f}"
def _bar(fraction: float, width: int = 30) -> str:
    filled = int(round(width * fraction))
    return "[" + "█" * filled + "─" * (width - filled) + "]"
def _hline(char: str = "─", width: int = 70) -> str: return char * width

@dataclass
class EpochRecord:
    fold: int; epoch: int; total_epochs: int
    train_loss: float = 0.0; train_acc: float = 0.0
    class_loss: float = 0.0; domain_loss: float = 0.0
    domain_acc_src: float = 0.0; domain_acc_tgt: float = 0.0
    grad_norm: float = 0.0; alpha: float = 0.0; lr: float = 0.0
    imgs_per_sec: float = 0.0; gpu_mem_mb: float = 0.0
    val_loss: float = 0.0; val_acc: float = 0.0
    val_auc: float = 0.0; val_f1: float = 0.0
    val_sensitivity: float = 0.0; val_specificity: float = 0.0; val_precision: float = 0.0
    epoch_secs: float = 0.0; is_best: bool = False

@dataclass
class FoldRecord:
    fold: int
    best_epoch: int = 0; best_val_auc: float = 0.0
    accuracy: float = 0.0; precision: float = 0.0; recall: float = 0.0
    specificity: float = 0.0; f1: float = 0.0; auc_roc: float = 0.0
    tp: int = 0; tn: int = 0; fp: int = 0; fn: int = 0
    epochs: list = field(default_factory=list)

class TrainingLogger:
    OVERFITTING_GAP = 0.10
    EMA_ALPHA       = 0.3

    def __init__(self, out_dir: str, n_folds: int, total_epochs: int):
        self.out_dir = out_dir
        self.n_folds = n_folds
        self.total_epochs = total_epochs
        self.json_path = os.path.join(out_dir, "training_journal.json")

        self.fold_records: list[FoldRecord] = []
        self._cur_fold: Optional[FoldRecord] = None
        self._cur_epoch: Optional[EpochRecord] = None

        self._batch_losses = []; self._batch_class_losses = []; self._batch_dom_losses = []
        self._batch_grad_norms = []; self._batch_dom_correct_src = []; self._batch_dom_correct_tgt = []
        self._batch_alphas = []; self._batch_n = []
        
        self._ema_epoch_secs: Optional[float] = None
        self._overall_start: float = time.time()

        os.makedirs(out_dir, exist_ok=True)
        log.info(_hline("═"))
        log.info(f"  TrainingLogger initialised  │  {n_folds} folds × {total_epochs} epochs")
        log.info(f"  Journal → {self.json_path}")
        log.info(_hline("═"))

    def begin_fold(self, fold: int, n_train: int, n_val: int):
        self._cur_fold = FoldRecord(fold=fold)
        log.info(f"\n{_hline('═')}\n  {_B}FOLD {fold}/{self.n_folds}{_W}  │  Train={n_train:,}  │  Val={n_val:,}\n{_hline('═')}")

    def end_fold(self, metrics: dict, val_labels: np.ndarray, val_probs: np.ndarray):
        r = self._cur_fold
        r.accuracy, r.precision, r.recall = metrics["accuracy"], metrics["precision"], metrics["recall"]
        r.specificity, r.f1, r.auc_roc = metrics["specificity"], metrics["f1"], metrics["auc_roc"]
        preds = (val_probs >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(val_labels, preds, labels=[0,1]).ravel()
        r.tp, r.tn, r.fp, r.fn = int(tp), int(tn), int(fp), int(fn)
        self.fold_records.append(r)
        self._print_fold_summary(r, val_labels, val_probs)
        self._flush_json()

    def begin_epoch(self, fold: int, epoch: int):
        self._cur_epoch = EpochRecord(fold=fold, epoch=epoch, total_epochs=self.total_epochs)
        for lst in [self._batch_losses, self._batch_class_losses, self._batch_dom_losses, self._batch_grad_norms, 
                    self._batch_dom_correct_src, self._batch_dom_correct_tgt, self._batch_alphas, self._batch_n]: lst.clear()
        self._epoch_start = time.time(); self._epoch_imgs = 0

    def log_batch(self, total_loss, class_loss, domain_loss, grad_norm, alpha, n_src, src_domain_logits, tgt_domain_logits):
        with torch.no_grad():
            dom_acc_src = (torch.sigmoid(src_domain_logits.float()) < 0.5).float().mean().item()
            dom_acc_tgt = (torch.sigmoid(tgt_domain_logits.float()) >= 0.5).float().mean().item()
        self._batch_losses.append(total_loss); self._batch_class_losses.append(class_loss); self._batch_dom_losses.append(domain_loss)
        self._batch_grad_norms.append(grad_norm); self._batch_alphas.append(alpha)
        self._batch_dom_correct_src.append(dom_acc_src); self._batch_dom_correct_tgt.append(dom_acc_tgt)
        self._batch_n.append(n_src); self._epoch_imgs += n_src

    def end_epoch(self, train_loss, train_acc, val_loss, val_acc, val_metrics, lr, is_best):
        elapsed = time.time() - self._epoch_start
        self._ema_epoch_secs = elapsed if self._ema_epoch_secs is None else (self.EMA_ALPHA * elapsed + (1 - self.EMA_ALPHA) * self._ema_epoch_secs)
        gpu_mem = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

        r = self._cur_epoch
        r.train_loss, r.train_acc, r.val_loss, r.val_acc = train_loss, train_acc, val_loss, val_acc
        r.val_auc, r.val_f1, r.lr, r.is_best, r.epoch_secs = val_metrics.get("auc_roc", 0.0), val_metrics.get("f1", 0.0), lr, is_best, elapsed
        r.val_sensitivity, r.val_specificity, r.val_precision = val_metrics.get("recall", 0.0), val_metrics.get("specificity", 0.0), val_metrics.get("precision", 0.0)
        r.imgs_per_sec, r.gpu_mem_mb = self._epoch_imgs / max(elapsed, 1e-6), gpu_mem
        r.alpha = float(np.mean(self._batch_alphas)) if self._batch_alphas else 0.0
        r.grad_norm = float(np.mean(self._batch_grad_norms)) if self._batch_grad_norms else 0.0
        r.class_loss = float(np.mean(self._batch_class_losses)) if self._batch_class_losses else 0.0
        r.domain_loss = float(np.mean(self._batch_dom_losses)) if self._batch_dom_losses else 0.0
        r.domain_acc_src = float(np.mean(self._batch_dom_correct_src)) if self._batch_dom_correct_src else 0.0
        r.domain_acc_tgt = float(np.mean(self._batch_dom_correct_tgt)) if self._batch_dom_correct_tgt else 0.0

        if self._cur_fold:
            self._cur_fold.epochs.append(asdict(r))
            if is_best: self._cur_fold.best_epoch, self._cur_fold.best_val_auc = r.epoch + 1, r.val_auc

        self._print_epoch_row(r)
        return r

    def print_epoch_header(self):
        log.info(_hline())
        log.info(f"  {'Ep':>4}  {'Progress':30}  {'TrLoss':>8}  {'TrAcc':>6}  {'VlLoss':>8}  {'VlAcc':>6}  {'AUC':>6}  {'F1':>6}  {'Sens':>6}  {'Spec':>6}  {'α':>5}  {'GNorm':>7}  {'DomSrc':>7}  {'DomTgt':>7}  {'LR':>9}  {'ImgS':>7}  {'GPU MB':>7}  {'ETA':>8}")
        log.info(_hline())

    def _print_epoch_row(self, r: EpochRecord):
        pct = ((r.epoch + 1) / r.total_epochs) * 100
        epochs_left = r.total_epochs - (r.epoch + 1)
        eta_str = f"{int((epochs_left * self._ema_epoch_secs)//60)}:{int((epochs_left * self._ema_epoch_secs)%60):02d}" if self._ema_epoch_secs and epochs_left > 0 else "  done  "
        auc_col = _G if r.val_auc >= 0.90 else (_Y if r.val_auc >= 0.80 else _R)
        dsrc_col = _G if abs(r.domain_acc_src - 0.5) < 0.15 else _Y
        dtgt_col = _G if abs(r.domain_acc_tgt - 0.5) < 0.15 else _Y
        best_tag = f" {_G}★ BEST{_W}" if r.is_best else ""

        log.info(
            f"  {r.epoch+1:>4}  {_bar(pct/100, 28)} {pct:5.1f}%  {r.train_loss:>8.4f}  {r.train_acc:>6.4f}  "
            f"{r.val_loss:>8.4f}  {r.val_acc:>6.4f}  {auc_col}{_fmt(r.val_auc)}{_W}  {_fmt(r.val_f1):>6}  "
            f"{_fmt(r.val_sensitivity):>6}  {_fmt(r.val_specificity):>6}  {r.alpha:>5.3f}  {r.grad_norm:>7.4f}  "
            f"{dsrc_col}{r.domain_acc_src:>7.4f}{_W}  {dtgt_col}{r.domain_acc_tgt:>7.4f}{_W}  "
            f"{r.lr:>9.2e}  {r.imgs_per_sec:>7.1f}  {r.gpu_mem_mb:>7.0f}  {eta_str:>8}{best_tag}"
        )
        gap = r.train_acc - r.val_acc
        if gap > self.OVERFITTING_GAP: log.warning(f"  {_Y}⚠  Overfitting signal: train_acc − val_acc = {gap:.3f}{_W}")

    def log_best_model(self, fold: int, epoch: int, auc: float, save_path: str):
        log.info(f"\n  {_G}{'━'*60}\n  {_G}  ★  NEW BEST MODEL — Fold {fold}  │  Epoch {epoch}  │  AUC = {auc:.6f}\n  {_G}{'━'*60}\n")

    def _print_fold_summary(self, r: FoldRecord, val_labels, val_probs):
        log.info(f"\n{_hline('═')}\n  {_B}FOLD {r.fold} SUMMARY{_W}  │  Best epoch: {r.best_epoch}  │  Best AUC: {r.best_val_auc:.6f}\n{_hline('─')}")
        log.info(f"  Confusion Matrix (threshold=0.5):\n           Pred No-Tumor  Pred Tumor\n  Act No-Tumor   TN={r.tn:>6}    FP={r.fp:>6}\n  Act Tumor      FN={r.fn:>6}    TP={r.tp:>6}\n{_hline('═')}")

    def print_cv_summary(self):
        if not self.fold_records: return
        log.info(f"\n{_hline('═')}\n  {_B}CROSS-VALIDATION FINAL REPORT{_W}\n{_hline('═')}")
        best_fold = max(self.fold_records, key=lambda r: r.auc_roc)
        log.info(f"  {_G}★ Best fold: Fold {best_fold.fold}  │  AUC={best_fold.auc_roc:.6f}  │  Best epoch: {best_fold.best_epoch}{_W}\n{_hline('═')}")
        self._flush_json()

    def _flush_json(self):
        tmp = self.json_path + ".tmp"
        with open(tmp, "w") as f: json.dump({"folds": [asdict(r) for r in self.fold_records]}, f, indent=2)
        os.replace(tmp, self.json_path)

---
# 7. Training & Evaluation Functions <a id='7'></a>

The training loop is now wired to the `TrainingLogger` at every granularity level:

- **Per-batch**: loss decomposition, gradient L2 norm, alpha, domain classifier accuracy
- **Per-epoch**: throughput (imgs/sec), GPU memory peak, ETA, overfitting gap detector
- **Per-fold**: ROC curve saved to disk, confusion matrix, per-metric coloured bar chart in log


In [24]:

# ==========================================
# 7. Training & Evaluation (Logger-Wired)
# ==========================================
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---> ADD THIS LINE <---
from typing import Optional
def _compute_grad_norm(model: nn.Module) -> float:
    """
    Computes the global L2 norm of all gradients.
    Called AFTER scaler.unscale_() so we see true (unscaled) gradient magnitudes.
    A healthy norm: stable and not exploding. Values > 10 with clipping = frequent clipping.
    """
    total_sq = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_sq += p.grad.detach().data.norm(2).item() ** 2
    return math.sqrt(total_sq)


def train_dann_epoch(
    model:          DANN_ResNet18,
    source_loader:  DataLoader,
    target_iter,
    optimizer:      optim.Optimizer,
    scaler:         torch.amp.GradScaler,
    device:         torch.device,
    current_epoch:  int,
    total_epochs:   int,
    grad_clip_norm: float = 1.0,
    logger:         Optional[TrainingLogger] = None,
) -> tuple[float, float]:
    """
    One DANN training epoch — fully wired to TrainingLogger.

    Returns (mean_total_loss, mean_class_accuracy)
    """
    model.train()
    loss_sum, correct_class, total = 0.0, 0, 0
    len_dataloader = len(source_loader)

    pbar = tqdm(
        enumerate(source_loader),
        total=len_dataloader,
        desc=f"  Train ep {current_epoch+1:02d}/{total_epochs}",
        leave=False,
        ncols=110,
    )

    for i, (src_imgs, src_labels) in pbar:
        tgt_imgs, _ = next(target_iter)

        src_imgs   = src_imgs.to(device, non_blocking=True)
        src_labels = src_labels.float().unsqueeze(1).to(device, non_blocking=True)
        tgt_imgs   = tgt_imgs.to(device, non_blocking=True)

        # ── Alpha schedule ────────────────────────────────────
        global_step = float(i + current_epoch * len_dataloader)
        total_steps = float(total_epochs * len_dataloader)
        p     = global_step / total_steps
        alpha = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            src_class_logits, src_domain_logits = model(src_imgs, alpha)
            src_class_loss   = CLASS_CRITERION(src_class_logits, src_labels)
            src_domain_labels = torch.zeros_like(src_domain_logits)
            src_domain_loss   = DOMAIN_CRITERION(src_domain_logits, src_domain_labels)

            _, tgt_domain_logits = model(tgt_imgs, alpha)
            tgt_domain_labels    = torch.ones_like(tgt_domain_logits)
            tgt_domain_loss      = DOMAIN_CRITERION(tgt_domain_logits, tgt_domain_labels)

            total_domain_loss = src_domain_loss + tgt_domain_loss
            total_loss        = src_class_loss + total_domain_loss

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = _compute_grad_norm(model)   # measured AFTER unscale, BEFORE clip
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        # ── Batch accounting ──────────────────────────────────
        n_src    = src_imgs.size(0)
        loss_sum += total_loss.item() * n_src
        preds     = (torch.sigmoid(src_class_logits) >= 0.5).long()
        correct_class += (preds == src_labels.long()).sum().item()
        total         += n_src

        # ── Logger: batch signals ─────────────────────────────
        if logger is not None:
            logger.log_batch(
                total_loss=total_loss.item(),
                class_loss=src_class_loss.item(),
                domain_loss=total_domain_loss.item(),
                grad_norm=grad_norm,
                alpha=alpha,
                n_src=n_src,
                src_domain_logits=src_domain_logits.detach(),
                tgt_domain_logits=tgt_domain_logits.detach(),
            )

        pbar.set_postfix({
            "TotL": f"{total_loss.item():.3f}",
            "ClsL": f"{src_class_loss.item():.3f}",
            "DomL": f"{total_domain_loss.item():.3f}",
            "Acc":  f"{(preds == src_labels.long()).float().mean().item():.3f}",
            "GN":   f"{grad_norm:.3f}",
            "α":    f"{alpha:.3f}",
        })

    pbar.close()
    return loss_sum / max(total, 1), correct_class / max(total, 1)


@torch.no_grad()
def evaluate(
    model:  DANN_ResNet18,
    loader: DataLoader,
    device: torch.device,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    """Inference-mode evaluation. Returns (loss, acc, labels, probs)."""
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_labels, all_probs   = [], []

    pbar = tqdm(loader, desc="  Evaluate", leave=False, ncols=80)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss   = CLASS_CRITERION(logits, labels)

        probs     = torch.sigmoid(logits.float())
        loss_sum += loss.item() * imgs.size(0)
        correct  += ((probs >= 0.5).long() == labels.long()).sum().item()
        total    += imgs.size(0)
        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())

    model.train()
    return (
        loss_sum / max(total, 1),
        correct  / max(total, 1),
        np.array(all_labels),
        np.array(all_probs),
    )
def compute_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:
    """
    Computes the full clinical metric suite appropriate for medical imaging.

    Metrics chosen for this task:
        - Sensitivity (Recall): Catches as many real tumours as possible. High priority.
        - Specificity:          Avoids false positives (unnecessary follow-ups/anxiety).
        - AUC-ROC:              Threshold-independent performance — most reliable single
                                metric for imbalanced medical datasets.
        - F1:                   Balances precision and recall; useful for reporting.
    """
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),      # = Sensitivity
        "specificity": float(tn / (tn + fp + 1e-8)),
        "f1":          f1_score(labels, preds, zero_division=0),
        "auc_roc":     roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0,
    }


---
# 8. Cross-Validation Runner <a id='8'></a>

The CV runner now passes a `TrainingLogger` instance through every level:
`run_dann_cv` → `begin_fold` → loop → `begin_epoch` → `train_dann_epoch` → `end_epoch` → `end_fold` → `print_cv_summary`.


In [25]:
# ==========================================
# 8. DANN Cross-Validation (Logger-Wired)
# ==========================================
import copy
import itertools

def run_dann_cv(brisc_records, figshare_records, device, config, brisc_cache=None, figshare_cache=None):
    labels_arr = np.array([r["label"] for r in brisc_records])
    groups_arr = np.array([r["patient_id"] for r in brisc_records])

    sgkf = StratifiedGroupKFold(n_splits=config["n_folds"], shuffle=True, random_state=config["seed"])

    # ── Initialise Logger Once ───────────
    logger = TrainingLogger(out_dir=config["out_dir"], n_folds=config["n_folds"], total_epochs=config["num_epochs"])

    # ── Target DataLoader (Shared across folds)
    target_ds = BrainMRIDataset(figshare_records, build_transforms(augment=True), cache=figshare_cache)
    target_dl = DataLoader(target_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True)

    fold_results = []
    best_auc, best_model = -1.0, None

    for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(brisc_records, labels_arr, groups=groups_arr), start=1):
        train_recs = [brisc_records[i] for i in tr_idx]
        val_recs   = [brisc_records[i] for i in vl_idx]

        # ── Logger: Fold Start ────────────────────────────────
        logger.begin_fold(fold=fold, n_train=len(train_recs), n_val=len(val_recs))

        train_ds = BrainMRIDataset(train_recs, build_transforms(augment=True), cache=brisc_cache)
        val_ds   = BrainMRIDataset(val_recs, build_transforms(augment=False), cache=brisc_cache)
        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True)
        val_dl   = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"])

        target_iter = itertools.cycle(target_dl)

        model     = DANN_ResNet18().to(device)
        optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        
        # ── Warmup + Cosine Annealing Scheduler ───────────────
        warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=config["warmup_epochs"])
        cosine_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["num_epochs"] - config["warmup_epochs"])
        scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[config["warmup_epochs"]])

        scaler = torch.amp.GradScaler(device="cuda", enabled=(device.type == "cuda"))

        best_vl_auc, best_wts = -1.0, None

        logger.print_epoch_header()

        for epoch in range(config["num_epochs"]):
            # ── Logger: Epoch Start ───────────────────────────
            logger.begin_epoch(fold=fold, epoch=epoch)

            tl, ta = train_dann_epoch(
                model=model, 
                source_loader=train_dl, 
                target_iter=target_iter, 
                optimizer=optimizer, 
                scaler=scaler,
                device=device, 
                current_epoch=epoch, 
                total_epochs=config["num_epochs"],
                grad_clip_norm=config.get("grad_clip_norm", 1.0),
                logger=logger
            )

            val_loss, val_acc, val_labels, val_probs = evaluate(model, val_dl, device)
            scheduler.step()

            ep_metrics = compute_metrics(val_labels, val_probs)
            ep_auc     = ep_metrics["auc_roc"]
            lr_now     = optimizer.param_groups[0]["lr"]
            is_best    = bool(ep_auc > best_vl_auc)

            if is_best:
                best_vl_auc = ep_auc
                best_wts    = copy.deepcopy(model.state_dict())
                save_path = os.path.join(config["out_dir"], "best_dann_brisc.pth")
                torch.save(best_wts, save_path)
                logger.log_best_model(fold=fold, epoch=epoch+1, auc=best_vl_auc, save_path=save_path)

            # ── Logger: Epoch End ─────────────────────────────
            logger.end_epoch(train_loss=tl, train_acc=ta, val_loss=val_loss, val_acc=val_acc, val_metrics=ep_metrics, lr=lr_now, is_best=is_best)

        # ── Fold Finalisation ─────────────────────────────────
        model.load_state_dict(best_wts)
        _, _, fl, fp = evaluate(model, val_dl, device)
        final_metrics = compute_metrics(fl, fp)
        final_metrics["fold"] = fold
        fold_results.append(final_metrics)

        # ── Logger: Fold End ──────────────────────────────────
        logger.end_fold(metrics=final_metrics, val_labels=fl, val_probs=fp)

        if final_metrics["auc_roc"] > best_auc:
            best_auc   = final_metrics["auc_roc"]
            best_model = copy.deepcopy(model)

    # ── Logger: Final Run Summary ─────────────────────────────
    logger.print_cv_summary()

    return best_model, fold_results

---
# 9. Master Execution <a id='9'></a>

In [26]:

# ==========================================
# 9. Master Execution
# ==========================================

def set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    log.info(f"Seeds set → {seed}  |  CuDNN deterministic: ON")


def main():
    os.makedirs(CONFIG["out_dir"], exist_ok=True)
    set_seeds(CONFIG["seed"])

    log.info("Downloading / locating datasets via KaggleHub...")
    import kagglehub
    brisc_path    = kagglehub.dataset_download("briscdataset/brisc2025")
    figshare_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

    brisc_records    = load_brisc(brisc_path,    slices_per_patient=CONFIG["slices_per_patient"])
    figshare_records = load_figshare(figshare_path)

    if not brisc_records or not figshare_records:
        log.error("Dataset loading failed — check paths above."); return

    log.info("\n── Checking preprocessing caches ──")
    brisc_cache    = build_cache(brisc_records,    cache_path=CONFIG["brisc_cache_path"])
    figshare_cache = build_cache(figshare_records, cache_path=CONFIG["figshare_cache_path"])

    best_ckpt = os.path.join(CONFIG["out_dir"], "best_dann_brisc.pth")
    if CONFIG.get("skip_training") and os.path.exists(best_ckpt):
        log.info(f">>> skip_training=True — model exists at {best_ckpt}. Skipping. <<<"); return

    log.info("\n>>> Starting DANN 5-Fold Cross-Validation <<<")
    best_model, cv_results = run_dann_cv(
        brisc_records, figshare_records, device, CONFIG,
        brisc_cache=brisc_cache, figshare_cache=figshare_cache,
    )
    log.info(f"\nPipeline complete! Best model → {best_ckpt}")


if __name__ == "__main__":
    main()


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
